# Chapitre 4 — Intrication et test de Bell

On construit $|\Phi^+\rangle$, on calcule son état réduit et son entropie, puis on évalue $S$ (CHSH) : $2\sqrt2$ pour l'état quantique, au plus $2$ pour toute stratégie locale déterministe.

*Carnet PHY321 — fonctionne sans connexion : numpy et matplotlib suffisent.*

In [1]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(321)  # graine fixe : résultats reproductibles

In [2]:
phi_plus = np.array([1, 0, 0, 1]) / np.sqrt(2)
rho = np.outer(phi_plus, phi_plus.conj()).reshape(2, 2, 2, 2)
rho_A = np.einsum('ijkj->ik', rho)                       # trace partielle sur B
w = np.linalg.eigvalsh(rho_A); w = w[w > 1e-12]
print("rho_A =", rho_A.real.tolist()); print("entropie de von Neumann (bits) =", -(w*np.log2(w)).sum())

rho_A = [[0.4999999999999999, 0.0], [0.0, 0.4999999999999999]]
entropie de von Neumann (bits) = 1.0


In [3]:
sx = np.array([[0,1],[1,0]]); sz = np.array([[1,0],[0,-1]])
def obs(a): return np.cos(a)*sz + np.sin(a)*sx
def E(a, b): return np.real(phi_plus @ np.kron(obs(a), obs(b)) @ phi_plus)
a, a2, b, b2 = 0, np.pi/2, np.pi/4, -np.pi/4
S_q = E(a, b) + E(a, b2) + E(a2, b) - E(a2, b2)
print("S_quantique =", S_q, "  2*sqrt(2) =", 2*np.sqrt(2))
# stratégies locales déterministes : A(a),A(a2),B(b),B(b2) ∈ {±1}
best = max(abs(A1*B1 + A1*B2 + A2*B1 - A2*B2) for A1 in (-1,1) for A2 in (-1,1) for B1 in (-1,1) for B2 in (-1,1))
print("S maximal, stratégie locale =", best)

S_quantique = 2.82842712474619   2*sqrt(2) = 2.8284271247461903
S maximal, stratégie locale = 2


## Échantillonnage de mesures

On tire des couples de résultats selon la règle de Born et on retrouve $E(a,b)=\cos(a-b)$.

In [4]:
def sample_E(a, b, N=20000):
    A, Bm = obs(a), obs(b)
    _, va = np.linalg.eigh(A); _, vb = np.linalg.eigh(Bm)
    basis = np.array([np.kron(va[:, i], vb[:, j]) for i in range(2) for j in range(2)])  # résultats (-1,-1),(-1,+1),(+1,-1),(+1,+1)
    probs = np.abs(basis.conj() @ phi_plus)**2
    signs = np.array([-1*-1, -1*1, 1*-1, 1*1])
    return rng.choice(signs, size=N, p=probs/probs.sum()).mean()
print(sample_E(a, b), np.cos(a-b)); print(sample_E(a2, b2), np.cos(a2-b2))

0.7035 0.7071067811865476
-0.712 -0.7071067811865475
